## Diagnostic d'alignement des lemmes TL/DMF (reprise du projet)

Ce notebook sert à :

1. Choisir un fichier TSV du corpus OF3C
2. Extraire les lemmes TL présents et les comparer au référentiel d'alignement déjà construit (`documentation/data_df_lemmes.tsv`).
3. Produire des rapports (lemmes absents, ambiguïtés) dans `documentation/diagnostics/`.

Les TSV OF3C ont au moins les colonnes `form`, `lemma`, `POS`.

In [ ]:
from pathlib import Path
import pandas as pd
import re
import datetime

# Détection automatique de la racine du dépôt
def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    markers = {"OF3C-main", "alignementReferentiels", "documentation", "scripts"}
    for p in [start, *start.parents]:
        if markers.issubset({c.name for c in p.iterdir()}):
            return p
    raise RuntimeError("Détection impossible. Lancer le notebook depuis le dossier du projet.")

REPO = find_repo_root(Path.cwd())
REPO


In [ ]:
DOC = REPO / "documentation"
SCRIPTS = REPO / "scripts"
OF3C = REPO / "OF3C-main"
ALIGN = REPO / "alignementReferentiels"

LEMMA_REF = DOC / "data_df_lemmes.tsv"  # pivot TL/DMF (+ autres)
ERRORS_ODT = DOC / "repertoireErreurs.odt"

OUTDIR = DOC / "diagnostics"
OUTDIR.mkdir(exist_ok=True)

print("REPO:", REPO)
print("LEMMA_REF exists:", LEMMA_REF.exists())
print("ERRORS_ODT exists:", ERRORS_ODT.exists())
print("OUTDIR:", OUTDIR)


In [ ]:
# Liste des TSV disponibles
# Corpus (OF3C) : plusieurs sous-dossiers possibles.
# On liste large : tout .tsv dans OF3C-main/tsv
tsv_paths = sorted((OF3C / "tsv").rglob("*.tsv"))
print(f"{len(tsv_paths)} fichiers TSV trouvés sous {OF3C/'tsv'}")
for i, p in enumerate(tsv_paths[:30]):
    print(f"{i:>2}  {p.relative_to(REPO)}")
if len(tsv_paths) > 30:
    print("... (affichage limité à 30)")


## Choisir le fichier à analyser

 Pour tester rapidement, utiliser le TSV déjà dans `documentation/` :  
  `documentation/nca-sample-naomicorr-corrige.tsv`.


In [ ]:
# Choisis un TSV :
# Option 1 : un TSV dans OF3C-main/tsv
# TSV_PATH = tsv_paths[0]

# Option 2 : l'échantillon déjà corrigé
# TSV_PATH = DOC / "nca-sample-naomicorr-corrige.tsv"

TSV_PATH = tsv_paths[6]

TSV_PATH = TSV_PATH.resolve()
print("TSV_PATH:", TSV_PATH)
print("Existe ?", TSV_PATH.exists())


In [ ]:
def read_tsv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False)
    # Nettoyage léger
    df.columns = [c.strip() for c in df.columns]
    for c in df.columns:
        df[c] = df[c].astype(str).str.strip()
    return df

df = read_tsv(TSV_PATH)
df.head(), df.shape, df.columns


In [ ]:
# Détection colonnes utiles
def pick_col(df: pd.DataFrame, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    # fallback : match insensible à la casse
    lower = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower:
            return lower[c.lower()]
    return None

COL_FORM = pick_col(df, ["form", "token", "word", "forme"])
COL_LEMMA = pick_col(df, ["lemma", "lemme"])
COL_POS = pick_col(df, ["POS", "pos", "cattex", "upos"])
COL_MORPH = pick_col(df, ["morph", "morphology", "traits", "feats"])

COL_FORM, COL_LEMMA, COL_POS, COL_MORPH


In [ ]:
# Stats rapides
if COL_LEMMA is None:
    raise ValueError(f"Colonne de lemme non trouvée dans {TSV_PATH.name}. Colonnes: {list(df.columns)}")

n_tokens = len(df)
n_lemmas = df[COL_LEMMA].nunique(dropna=False)
n_pos = df[COL_POS].nunique(dropna=False) if COL_POS else None

print("Tokens:", n_tokens)
print("Lemmes (distincts):", n_lemmas)
print("POS (distincts):", n_pos)


In [ ]:
# Chargement référentiel TL/DMF
ref = pd.read_csv(LEMMA_REF, sep="\t", dtype=str, keep_default_na=False)
ref.columns = [c.strip() for c in ref.columns]
for c in ref.columns:
    ref[c] = ref[c].astype(str).str.strip()

# Colonnes attendues : TL, DMF, ...
if "TL" not in ref.columns or "DMF" not in ref.columns:
    raise ValueError(f"Colonnes TL/DMF non trouvées dans {LEMMA_REF}. Colonnes: {list(ref.columns)}")

ref.head(), ref.shape, ref.columns


In [ ]:
# Mapping TL vers DMF (avec gestion des multi-correspondances)
# Convention : DMF peut contenir plusieurs entrées séparées par '+'ou ';' selon les cas
def split_multi(s: str):
    s = (s or "").strip()
    if not s:
        return []
    # séparateurs fréquents
    parts = re.split(r"\s*\+\s*|\s*;\s*|\s*,\s*", s)
    return [p for p in (p.strip() for p in parts) if p]

ref["DMF_list"] = ref["DMF"].apply(split_multi)
ref["DMF_n"] = ref["DMF_list"].apply(len)

tl_to_dmf = dict(zip(ref["TL"], ref["DMF"]))
tl_set = set(ref["TL"])

print("TL uniques dans le référentiel:", len(tl_set))
print("Entrées TL avec DMF vide:", int((ref["DMF"].str.strip() == "").sum()))
print("Entrées TL avec DMF multiple:", int((ref["DMF_n"] > 1).sum()))


In [ ]:
# Diagnostic principal : lemme du corpus présent/absent du référentiel
lemmas = df[COL_LEMMA].fillna("").astype(str).str.strip()
lemma_counts = lemmas.value_counts(dropna=False).rename_axis("TL").reset_index(name="token_count")

lemma_counts["in_ref"] = lemma_counts["TL"].isin(tl_set)
lemma_counts["DMF"] = lemma_counts["TL"].map(tl_to_dmf).fillna("")

# Flags utiles
lemma_counts["missing_in_ref"] = ~lemma_counts["in_ref"]
lemma_counts["dmf_empty"] = lemma_counts["in_ref"] & (lemma_counts["DMF"].str.strip() == "")
lemma_counts["dmf_multi"] = lemma_counts["DMF"].str.contains(r"\+|;|,", regex=True)

summary = {
    "tokens_total": int(n_tokens),
    "lemmas_distinct": int(n_lemmas),
    "lemmas_missing_in_ref": int(lemma_counts["missing_in_ref"].sum()),
    "lemmas_in_ref_dmf_empty": int(lemma_counts["dmf_empty"].sum()),
    "lemmas_in_ref_dmf_multi": int(lemma_counts["dmf_multi"].sum()),
}

summary


In [ ]:
# Exports : trois rapports
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
base = f"{TSV_PATH.stem}__{stamp}"

missing = lemma_counts[lemma_counts["missing_in_ref"]].copy()
dmf_empty = lemma_counts[lemma_counts["dmf_empty"]].copy()
dmf_multi = lemma_counts[lemma_counts["dmf_multi"]].copy()

missing_path = OUTDIR / "missing_in_ref.tsv"
empty_path = OUTDIR / "dmf_empty.tsv"
multi_path = OUTDIR / "dmf_multi.tsv"
summary_path = OUTDIR / "summary.json"

missing.to_csv(missing_path, sep="\t", index=False)
dmf_empty.to_csv(empty_path, sep="\t", index=False)
dmf_multi.to_csv(multi_path, sep="\t", index=False)

import json
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Écrits :")
print(" -", missing_path.relative_to(REPO))
print(" -", empty_path.relative_to(REPO))
print(" -", multi_path.relative_to(REPO))
print(" -", summary_path.relative_to(REPO))


In [ ]:
# Aperçu rapide : top 30 lemmes absents
missing.head(30)
